# 08.6c AudioLDM2 推理

AudioLDM2 代表 latent diffusion 的文本到音频路线。依赖和权重可用时，本 Notebook 会直接加载模型并生成音频；缺少依赖时会打印安装指引。


## 运行环境与安装

在 Jupyter 中选择 kernel：`Python 3.11 (chapter08-diffusers)`。首次运行前先建立独立环境。mac/Linux 终端使用：

Hugging Face 当前 CLI 命令名为 `hf`。若 `hf --help` 不可用，可按官方文档先安装 standalone CLI；mac/Linux 使用 `curl -LsSf https://hf.co/cli/install.sh | bash`，Windows PowerShell 使用 `powershell -ExecutionPolicy ByPass -c "irm https://hf.co/cli/install.ps1 | iex"`。也可以把下面的 `hf download ...` 改成 `uvx hf download ...`。

```bash
cd CODE
python3.11 -m venv venv_ch08_diffusers
source venv_ch08_diffusers/bin/activate
python -m pip install --upgrade pip setuptools wheel
python -m pip install "torch==2.4.1" "torchaudio==2.4.1" soundfile scipy pandas librosa ipykernel ipywidgets
python -m pip install "diffusers==0.35.2" "transformers==4.49.0" "accelerate>=0.31,<2" "huggingface_hub>=0.23,<1" safetensors sentencepiece
python -m ipykernel install --user --name chapter08-diffusers --display-name "Python 3.11 (chapter08-diffusers)"
```

Windows PowerShell 使用：

```powershell
cd CODE
py -3.11 -m venv venv_ch08_diffusers
.\venv_ch08_diffusers\Scripts\Activate.ps1
python -m pip install --upgrade pip setuptools wheel
python -m pip install "torch==2.4.1" "torchaudio==2.4.1" soundfile scipy pandas librosa ipykernel ipywidgets
python -m pip install "diffusers==0.35.2" "transformers==4.49.0" "accelerate>=0.31,<2" "huggingface_hub>=0.23,<1" safetensors sentencepiece
python -m ipykernel install --user --name chapter08-diffusers --display-name "Python 3.11 (chapter08-diffusers)"
```

本章的 `chapter08-diffusers` kernel 服务 AudioLDM2。这里固定 Torch/Diffusers/Transformers 版本，避免 `pip install torch` 自动装到过新的运行时。Transformers 版本是为了兼容 AudioLDM2：它的 `language_model` 是 `GPT2Model`，Transformers 4.50+ 不再让这类非生成模型继承 generation helper，而 `diffusers.AudioLDM2Pipeline` 仍会调用这些 helper。若已经装到较新的 Torch 或 Transformers，在 `venv_ch08_diffusers` 中执行：

```bash
python -m pip install --force-reinstall "torch==2.4.1" "torchaudio==2.4.1" "diffusers==0.35.2" "transformers==4.49.0"
```

若已建立过 `venv_ch08_diffusers` 但缺少 AudioLDM2 依赖，在该环境中重新执行上方 pip 命令。


## 模型权重下载

mac/Linux 终端：

```bash
cd CODE
source venv_ch08_diffusers/bin/activate
hf download cvssp/audioldm2 --local-dir chapter08/models/cvssp_audioldm2
```

Windows PowerShell：

```powershell
cd CODE
.\venv_ch08_diffusers\Scripts\Activate.ps1
hf download cvssp/audioldm2 --local-dir chapter08/models/cvssp_audioldm2
```

下载后本 Notebook 会自动优先使用 `chapter08/models/cvssp_audioldm2`。


In [ ]:
from pathlib import Path
import os
import sys

# 路径推断：从 cwd 向上找含 CODE/chapter08/_common 的目录；ROOT 指向 CODE/chapter08/
_p = Path.cwd()
while not (_p / "CODE" / "chapter08" / "_common").exists():
    _parent = _p.parent
    if _parent == _p:
        raise FileNotFoundError("未找到项目根目录（包含 CODE/chapter08/_common 的目录），请在项目内运行本 Notebook")
    _p = _parent
ROOT = _p / "CODE" / "chapter08"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import Audio, display

from _common.config import load_yaml_config
from _common.device_utils import choose_device
from _common.paths import portable_path
from evaluation.comparison_table import append_model_comparison
from model_runners.base import GenerationRequest
from model_runners.conditioning import build_conditioning_rows

OUTPUT_TABLES = ROOT / "outputs" / "tables"
OUTPUT_TABLES.mkdir(parents=True, exist_ok=True)

def rel(path):
    return portable_path(path, ROOT)

def resolve_device(config=None):
    requested = os.getenv("CHAPTER08_DEVICE")
    if requested is None and config is not None:
        requested = str(config.get("device", "auto"))
    return choose_device(requested or "auto")

def print_setup_guidance(status):
    print(status.reason)
    if status.next_action:
        print(status.next_action)
    print("After completing the setup or moving to compatible hardware, rerun this Notebook; it will load and run the model directly.")

def print_runtime_guidance(error):
    print(str(error))
    print("Resolve the message above, then rerun this Notebook or the current cell.")

from model_runners.audioldm2 import AudioLDM2Runner

runner = AudioLDM2Runner()
status = runner.check_environment()
config = load_yaml_config(ROOT / "configs" / "audioldm2_inference.yaml")
display(pd.DataFrame([status.as_row()]))


In [ ]:
display(pd.DataFrame(build_conditioning_rows("audioldm2")))
condition_row = {
    "prompt": config["prompts"][0]["text"],
    "negative_prompt": config.get("negative_prompt", ""),
    "duration_seconds": config.get("duration_seconds", ""),
    "num_inference_steps": config.get("num_inference_steps", ""),
    "guidance_scale": config.get("guidance_scale", ""),
}
display(pd.DataFrame([condition_row]))


In [ ]:
if status.available:
    prompt = config["prompts"][0]
    request = GenerationRequest(
        prompt=prompt["text"],
        prompt_id=prompt["prompt_id"],
        duration_sec=float(config.get("duration_seconds", 8)),
        output_dir=ROOT / config["outputs"]["audio_dir"],
        extra={
            "model_name": os.getenv("CHAPTER08_AUDIOLDM2_MODEL", config.get("model_name", runner.model_name)),
            "device": resolve_device(config),
            "num_inference_steps": int(config.get("num_inference_steps", 25)),
            "guidance_scale": float(config.get("guidance_scale", 3.5)),
            "negative_prompt": config.get("negative_prompt", ""),
        },
    )
    try:
        result = runner.generate(request)
    except RuntimeError as exc:
        print_runtime_guidance(exc)
    else:
        append_model_comparison(
            ROOT / config["outputs"]["table_csv"],
            {
                "model_name": result.model_name,
                "prompt_id": result.prompt_id,
                "dataset_context": "text prompt plus negative prompt",
                "duration_sec": result.duration_sec,
                "wall_time_sec": result.wall_time_sec,
                "device": result.device,
                "output_audio_path": result.output_audio_path,
            },
        )
        display(Audio(str(result.output_audio_path)))
else:
    print_setup_guidance(status)
